# SGC на гетерофильных графах: подтверждение гипотезы

**Цель:** проверить, уступает ли простой low‑pass фильтр SGC (K=2) обучаемым агрегаторам (GCN, GAT) на датасетах с низкой гомофилией.

Используем два эталонных гетерофильных датасета из статьи «Large Scale Learning on Non‑Homophilous Graphs»:
- **Roman‑empire** (слова и части речи)
- **Amazon‑ratings** (товары и их рейтинги)

Ожидается, что фиксированное сглаживание SGC будет размывать границы классов, тогда как GCN и GAT с обучаемой агрегацией смогут выучить полезные паттерны.

In [129]:
#!pip install -q torch_geometric


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [130]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import HeterophilousGraphDataset
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.utils import add_self_loops, degree
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [131]:
torch.manual_seed(42)
np.random.seed(42)

## Вспомогательные функции

In [132]:
def compute_sgc_features(data, K=2):
    """Предвычисление S^K X (нормализованная матрица смежности с петлями)."""
    edge_index, num_nodes = data.edge_index, data.num_nodes
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
    row, col = edge_index
    deg = degree(row, num_nodes, dtype=torch.float)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

    adj = torch.zeros(num_nodes, num_nodes)
    adj[row, col] = norm
    x = data.x
    for _ in range(K):
        x = adj @ x
    return x

def normalize_features(x):
    """Стандартизация признаков (среднее 0, std 1)."""
    mean = x.mean(dim=0, keepdim=True)
    std = x.std(dim=0, keepdim=True) + 1e-8
    return (x - mean) / std

In [133]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8, dropout=0.6):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1,
                            concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [134]:
def train_gnn(model, data, train_mask, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_gnn(model, data, mask):
    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
        correct = (pred[mask] == data.y[mask]).sum().float()
        return (correct / mask.sum()).item()

In [135]:
def create_random_split(data, train_ratio=0.6, val_ratio=0.2,
                        test_ratio=0.2, random_state=None):
    num_nodes = data.num_nodes
    indices = np.arange(num_nodes)
    y = data.y.cpu().numpy()
    try:
        train_idx, test_val_idx = train_test_split(
            indices, test_size=val_ratio + test_ratio,
            random_state=random_state, stratify=y)
    except ValueError:
        train_idx, test_val_idx = train_test_split(
            indices, test_size=val_ratio + test_ratio,
            random_state=random_state)
    try:
        val_idx, test_idx = train_test_split(
            test_val_idx, test_size=test_ratio / (val_ratio + test_ratio),
            random_state=random_state, stratify=y[test_val_idx])
    except ValueError:
        val_idx, test_idx = train_test_split(
            test_val_idx, test_size=test_ratio / (val_ratio + test_ratio),
            random_state=random_state)
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    return train_mask, val_mask, test_mask

## Эксперимент

In [136]:
datasets_config = [
    ('Roman-empire', HeterophilousGraphDataset, '/tmp/Hetero', 'Roman-empire'),
    ('Amazon-ratings', HeterophilousGraphDataset, '/tmp/Hetero', 'Amazon-ratings')
    #,('Tolokers', HeterophilousGraphDataset, 'tmp/Hetero', 'Tolokers')
    #,('Minesweeper', HeterophilousGraphDataset, '/tmp/Hetero', 'Minesweeper')
]

results = []
MAX_EPOCHS = 200
PATIENCE = 20
LR = 0.01
WEIGHT_DECAY = 5e-4
NUM_SPLITS = 5
SGC_K = 2

for name, dataset_cls, root, ds_name in datasets_config:
    print(f'=== {name} ===')
    dataset = dataset_cls(root=root, name=ds_name)
    data = dataset[0]
    print(f'  Узлов: {data.num_nodes}, Рёбер: {data.num_edges}, Классов: {dataset.num_classes}')

    data.x = normalize_features(data.x)

    X_sgc_raw = compute_sgc_features(data, K=SGC_K)
    scaler = StandardScaler()
    X_sgc_np = scaler.fit_transform(X_sgc_raw.numpy())
    X_sgc = torch.from_numpy(X_sgc_np).float()

    acc_sgc, acc_gcn, acc_gat = [], [], []

    for split_seed in range(NUM_SPLITS):
        train_mask, val_mask, test_mask = create_random_split(data, random_state=split_seed)

        # ----- SGC -----
        X_train = X_sgc[train_mask].numpy()
        y_train = data.y[train_mask].numpy()
        X_test = X_sgc[test_mask].numpy()
        y_test = data.y[test_mask].numpy()
        clf = LogisticRegression(max_iter=1000, solver='lbfgs')
        clf.fit(X_train, y_train)
        acc_sgc.append(clf.score(X_test, y_test))

        # ----- GCN -----
        gcn = GCN(data.num_features, 128, dataset.num_classes, dropout=0.5)
        opt_gcn = torch.optim.Adam(gcn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()
        best_val = 0
        best_weights = None
        no_improve = 0
        for epoch in range(1, MAX_EPOCHS+1):
            train_gnn(gcn, data, train_mask, opt_gcn, criterion)
            val_acc = eval_gnn(gcn, data, val_mask)
            if val_acc > best_val:
                best_val = val_acc
                best_weights = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
        gcn.load_state_dict(best_weights)
        acc_gcn.append(eval_gnn(gcn, data, test_mask))

        # ----- GAT -----
        gat = GAT(data.num_features, 16, dataset.num_classes, heads=8, dropout=0.6)
        opt_gat = torch.optim.Adam(gat.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        best_val = 0
        best_weights = None
        no_improve = 0
        for epoch in range(1, MAX_EPOCHS+1):
            train_gnn(gat, data, train_mask, opt_gat, criterion)
            val_acc = eval_gnn(gat, data, val_mask)
            if val_acc > best_val:
                best_val = val_acc
                best_weights = {k: v.cpu().clone() for k, v in gat.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
        gat.load_state_dict(best_weights)
        acc_gat.append(eval_gnn(gat, data, test_mask))

    results.append({
        'Датасет': name,
        'SGC (mean ± std)': f"{np.mean(acc_sgc)*100:.2f} ± {np.std(acc_sgc)*100:.2f}",
        'GCN (mean ± std)': f"{np.mean(acc_gcn)*100:.2f} ± {np.std(acc_gcn)*100:.2f}",
        'GAT (mean ± std)': f"{np.mean(acc_gat)*100:.2f} ± {np.std(acc_gat)*100:.2f}"
    })
    print(f"  SGC: {np.mean(acc_sgc)*100:.2f} ± {np.std(acc_sgc)*100:.2f}")
    print(f"  GCN: {np.mean(acc_gcn)*100:.2f} ± {np.std(acc_gcn)*100:.2f}")
    print(f"  GAT: {np.mean(acc_gat)*100:.2f} ± {np.std(acc_gat)*100:.2f}")

=== Roman-empire ===
  Узлов: 22662, Рёбер: 65854, Классов: 18
  SGC: 40.83 ± 0.78
  GCN: 47.13 ± 0.51
  GAT: 49.07 ± 0.76
=== Amazon-ratings ===
  Узлов: 24492, Рёбер: 186100, Классов: 5
  SGC: 42.09 ± 0.57
  GCN: 49.22 ± 0.65
  GAT: 47.52 ± 1.05


## Результаты

In [137]:
results_df = pd.DataFrame(results)
results_df

,Датасет,SGC (mean ± std),GCN (mean ± std),GAT (mean ± std)
0,Roman-empire,40.83 ± 0.78,47.13 ± 0.51,49.07 ± 0.76
1,Amazon-ratings,42.09 ± 0.57,49.22 ± 0.65,47.52 ± 1.05


In [138]:
results_df.to_csv('heterophily_results.csv', index=False, encoding='utf-8-sig')
print('Сохранено в heterophily_results.csv')

Сохранено в heterophily_results.csv


## Выводы
- На **Roman‑empire** SGC уступает GCN и GAT на 6–9%.
- На **Amazon‑ratings** отставание составляет 5–7%.
- В обоих случаях обучаемые агрегации показывают значимо более высокую точность.
- Гипотеза о неуниверсальности SGC на гетерофильных графах подтверждается, хотя эффект сильно зависит от датасета: когда признаки узлов очень сильны (Minesweeper, Tolokers), SGC может проигрывать незначительно или не проигрывать вовсе (было проверено на датасетах Squirrel, Actor, Chameleon и меньших датасетах (Texas, Wisconsin, Cornell)).